In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import json
import os
from google.colab import drive
from sentence_transformers import SentenceTransformer

if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

data_path = '/content/drive/MyDrive/Aiproject/combined_dataset (1).json'
save_path = '/content/drive/MyDrive/Aiproject/psychology_model.pth'

model = SentenceTransformer('all-MiniLM-L6-v2')

contexts, responses = [], []
with open(data_path, 'r', encoding='utf-8') as f:
    for line in f:
        data = json.loads(line)
        contexts.append(data['Context'])
        responses.append(data['Response'])

all_embeddings = model.encode(contexts, convert_to_tensor=True, device=device)

class PsychologyNet(nn.Module):
    def __init__(self, input_size):
        super(PsychologyNet, self).__init__()
        self.fc = nn.Sequential(
            nn.Linear(input_size, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, input_size)
        )

    def forward(self, x):
        return self.fc(x)

input_dim = 384
net = PsychologyNet(input_dim).to(device)

if not os.path.exists(save_path):
    criterion = nn.MSELoss()
    optimizer = optim.Adam(net.parameters(), lr=0.001)

    for epoch in range(20):
        optimizer.zero_grad()
        outputs = net(all_embeddings)
        loss = criterion(outputs, all_embeddings)
        loss.backward()
        optimizer.step()
        print(f'Epoch [{epoch+1}/20], Loss: {loss.item():.4f}')

    torch.save(net.state_dict(), save_path)
else:
    net.load_state_dict(torch.load(save_path, map_location=device, weights_only=False))

net.eval()

def start_chat():
    print("\n--- Psychology AI Assistant Online ---")
    while True:
        user_input = input("You: ")
        if user_input.lower() in ['quit', 'exit', 'stop']:
            break

        with torch.no_grad():
            user_emb = model.encode([user_input], convert_to_tensor=True, device=device)
            predicted_vec = net(user_emb)
            cos_sim = nn.CosineSimilarity(dim=1)
            scores = cos_sim(all_embeddings, predicted_vec)
            best_idx = torch.argmax(scores).item()
            print(f"AI: {responses[best_idx]}\n")

start_chat()